In [1]:
from dotenv import  load_dotenv

load_dotenv()

True

In [2]:
import os
import json
import wikipedia
import psycopg2
from pgvector.psycopg2 import register_vector
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()

def get_connection():
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    return conn 

def create_tables():
    conn = get_connection()
    cursor = conn.cursor()
    
    cursor.execute("CREATE SCHEMA IF NOT EXISTS public")
    cursor.execute("SET search_path TO public")
    conn.commit()

    
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector")
    conn.commit()

    register_vector(conn)

    cursor.execute("CREATE SCHEMA IF NOT EXISTS badminton")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS badminton.singles_players (
            competitor_id   TEXT PRIMARY KEY,
            name            TEXT,
            country         TEXT,
            date_of_birth   TEXT,
            category        TEXT,
            rank            INTEGER,
            points          INTEGER
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS badminton.doubles_pairs (
            pair_id     TEXT PRIMARY KEY,
            pair_name   TEXT,
            category    TEXT,
            rank        INTEGER,
            points      INTEGER
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS badminton.doubles_players (
            id              SERIAL PRIMARY KEY,
            competitor_id   TEXT UNIQUE,
            name            TEXT,
            country         TEXT,
            date_of_birth   TEXT,
            pair_id         TEXT REFERENCES badminton.doubles_pairs(pair_id)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS badminton.player_profiles (
            id              SERIAL PRIMARY KEY,
            competitor_id   TEXT,
            name            TEXT,
            content_type    TEXT,
            content         TEXT,
            chunk_index     INTEGER,
            embedding       vector(1536)
        )
    """)

    cursor.execute("""
        CREATE INDEX IF NOT EXISTS player_profiles_embedding_idx
        ON badminton.player_profiles
        USING ivfflat (embedding vector_cosine_ops)
        WITH (lists = 100)
    """)
    
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_profiles_name
        ON badminton.player_profiles (name);        
    """)
    
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_profiles_content_type 
        ON badminton.player_profiles (content_type);
    """)
    
    conn.commit()
    cursor.close()
    conn.close()
    print("Tables created successfully")

In [3]:
SINGLES_CATEGORIES = [
    "bwf_men_singles_world_ranking",
    "bwf_women_singles_world_ranking"
]

DOUBLES_CATEGORIES = [
    "bwf_men_doubles_world_ranking",
    "bwf_women_doubles_world_ranking",
    "bwf_mixed_doubles_world_ranking"
]

def insert_singles(data, category):
    conn = get_connection()
    cursor = conn.cursor()
    count = 0

    for ranking_block in data.get("rankings", []):
        if ranking_block.get("name") != category:
            continue

        for entry in ranking_block.get("competitor_rankings", []):
            if entry.get("rank", 999) > 100:
                continue

            comp = entry.get("competitor", {})

            cursor.execute("""
                INSERT INTO badminton.singles_players
                    (competitor_id, name, country, date_of_birth, category, rank, points)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (competitor_id) DO UPDATE SET
                    rank     = EXCLUDED.rank,
                    points   = EXCLUDED.points,
                    category = EXCLUDED.category
            """, (
                comp.get("id"),
                comp.get("name"),
                comp.get("country"),
                comp.get("date_of_birth"),
                category,
                entry.get("rank"),
                entry.get("points")
            ))
            count += 1

    conn.commit()
    cursor.close()
    conn.close()
    print(f"Inserted {count} singles players for {category}")


In [4]:
def insert_doubles(data, category):
    conn = get_connection()
    cursor = conn.cursor()
    count = 0

    for ranking_block in data.get("rankings", []):
        if ranking_block.get("name") != category:
            continue

        for entry in ranking_block.get("competitor_rankings", []):
            if entry.get("rank", 999) > 100:
                continue

            comp = entry.get("competitor", {})
            players = comp.get("players", [])

            full_pair_name = " / ".join(p.get("name", "") for p in players) if players else comp.get("name")

            cursor.execute("""
                INSERT INTO badminton.doubles_pairs (pair_id, pair_name, category, rank, points)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (pair_id) DO UPDATE SET
                    rank      = EXCLUDED.rank,
                    points    = EXCLUDED.points,
                    category  = EXCLUDED.category,
                    pair_name = EXCLUDED.pair_name
            """, (
                comp.get("id"),
                full_pair_name,
                category,
                entry.get("rank"),
                entry.get("points")
            ))

            for player in players:
                cursor.execute("""
                    INSERT INTO badminton.doubles_players
                        (competitor_id, name, country, date_of_birth, pair_id)
                    VALUES (%s, %s, %s, %s, %s)
                    ON CONFLICT (competitor_id) DO NOTHING
                """, (
                    player.get("id"),
                    player.get("name"),
                    player.get("country"),
                    player.get("date_of_birth"),
                    comp.get("id")
                ))
            count += 1

    conn.commit()
    cursor.close()
    conn.close()
    print(f"Inserted {count} doubles pairs for {category}")

In [5]:

SECTION_KEYWORDS = {
    "biography": [
        "early life", "personal life", "background",
        "education", "childhood", "family", "life"
    ],
    "career": [
        "career", "professional", "junior", "senior",
        "playing style", "technique", "training"
    ],
    "achievement": [
        "achievement", "title", "championship", "award",
        "honours", "honor", "medal", "record", "accomplishment",
        "grand prix", "superseries", "world tour"
    ]
}

def extract_wiki_sections(player_name: str) -> dict:
    """
    Extracts biography, career, and achievement sections from Wikipedia.
    Returns only sections that exist — safely handles missing sections.
    """
    sections_found = {
        "biography": "",
        "career": "",
        "achievement": ""
    }

    try:
        page = wikipedia.page(player_name, auto_suggest=True)
    except wikipedia.DisambiguationError as e:
        print(f"Disambiguation for '{player_name}', trying: {e.options[0]}")
        try:
            page = wikipedia.page(e.options[0])
        except Exception as inner_e:
            print(f"Failed after disambiguation: {inner_e}")
            return sections_found
    except wikipedia.PageError:
        print(f"No Wikipedia page for '{player_name}'")
        return sections_found
    except Exception as e:
        print(f" Unexpected error fetching '{player_name}': {e}")
        return sections_found

    if page.summary.strip():
        sections_found["biography"] = page.summary
        print(f"biography summary found")
    else:
        print(f"no biography summary found")

    try:
        content = page.content

        raw_sections = content.split("\n== ")

        for raw_section in raw_sections:
            if not raw_section.strip():
                continue

            lines = raw_section.strip().split("\n")
            section_title = lines[0].replace("==", "").strip().lower()
            section_body = "\n".join(lines[1:]).strip()

            if not section_body:
                continue

            for content_type, keywords in SECTION_KEYWORDS.items():
                if content_type == "biography":
                    continue  
                if any(keyword in section_title for keyword in keywords):
                    sections_found[content_type] += f"{section_title.title()}:\n{section_body}\n\n"
                    break

    except Exception as e:
        print(f"Error parsing sections for '{player_name}': {e}")

    for content_type, content in sections_found.items():
        if content.strip():
            print(f"{content_type:<15} content found")
        else:
            print(f"{content_type:<15} not found, skipping")

    return sections_found


In [6]:
def normalize_name(name: str) -> str:
    if "," in name:
        parts = name.split(",")
        return f"{parts[1].strip()} {parts[0].strip()}"
    return name.strip()

In [7]:
embeddings_model = OpenAIEmbeddings()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

def store_player_profile(competitor_id: str, player_name: str):
    
    player_name = normalize_name(player_name)
    print(f"\n Adding {player_name} profile")

    sections = extract_wiki_sections(player_name)

    if not any(content.strip() for content in sections.values()):
        print(f"  No content found at all for {player_name}")
        return

    conn = get_connection()
    register_vector(conn) 
    cursor = conn.cursor()

    total_chunks = 0

    for content_type, content in sections.items():
        if not content.strip():
            continue

        chunks = text_splitter.split_text(content)

        for chunk_index, chunk in enumerate(chunks):
            if len(chunk.strip()) < 50:
                continue

            try:
                vector = embeddings_model.embed_query(chunk)

                cursor.execute("""
                    INSERT INTO badminton.player_profiles
                        (competitor_id, name, content_type, content, chunk_index, embedding)
                    VALUES (%s, %s, %s, %s, %s, %s)
                """, (
                    competitor_id,
                    player_name,
                    content_type,
                    chunk,
                    chunk_index,
                    vector
                ))
                total_chunks += 1

            except Exception as e:
                print(f" Failed embedding chunk {chunk_index} for {player_name}: {e}")
                continue

    conn.commit()
    cursor.close()
    conn.close()
    print(f" Stored {total_chunks} total chunks for {player_name}")


In [8]:
def fetch_all_profiles():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT competitor_id, name FROM badminton.singles_players WHERE name IS NOT NULL")
    singles = cursor.fetchall()

    cursor.execute("SELECT competitor_id, name FROM badminton.doubles_players WHERE name IS NOT NULL")
    doubles = cursor.fetchall()

    cursor.close()
    conn.close()

    seen = {}
    for competitor_id, name in singles + doubles:
        if competitor_id not in seen:
            seen[competitor_id] = name

    all_players = list(seen.items())
    print(f"\nFetching Wikipedia profiles for {len(all_players)} players...\n")

    success, skipped, failed = 0, 0, 0

    for competitor_id, name in all_players:
        try:
            store_player_profile(competitor_id, name)
            success += 1
        except Exception as e:
            print(f" Error processing {name}: {e}")
            failed += 1

    print(f"Done fetching player information, number of success is {success} and number of failed attempt is {failed}")

In [9]:
import requests

sportradar_api_key = os.getenv("SPORTRADAR_API_KEY")

def get_sportrader_info(url):
    
    headers = {
        "accept": "application/json",
        "x-api-key": sportradar_api_key
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        return {"error": "Failed to fetch competition data"}
    
    data = response.json()
    return data

url = "https://api.sportradar.com/badminton/trial/v2/en/rankings.json"

data = get_sportrader_info(url)

In [ ]:
create_tables()

for category in SINGLES_CATEGORIES:
    insert_singles(data, category)

Tables created successfully
Inserted 98 singles players for bwf_men_singles_world_ranking
Inserted 100 singles players for bwf_women_singles_world_ranking


In [11]:
for category in DOUBLES_CATEGORIES:
    insert_doubles(data, category)

Inserted 96 doubles pairs for bwf_men_doubles_world_ranking
Inserted 87 doubles pairs for bwf_women_doubles_world_ranking
Inserted 89 doubles pairs for bwf_mixed_doubles_world_ranking


In [12]:
fetch_all_profiles()

print("Database is filled")


Fetching Wikipedia profiles for 662 players...


 Adding Kunlavut Vitidsarn profile
biography summary found
biography       content found
career          content found
achievement     content found
 Stored 33 total chunks for Kunlavut Vitidsarn

 Adding Yu Qi Shi profile
biography summary found
biography       content found
career          content found
achievement     content found
 Stored 67 total chunks for Yu Qi Shi

 Adding Anders Antonsen profile
biography summary found
biography       content found
career          content found
achievement     content found
 Stored 26 total chunks for Anders Antonsen

 Adding Jonatan Christie profile
No Wikipedia page for 'Jonatan Christie'
  No content found at all for Jonatan Christie

 Adding Christo Popov profile
biography summary found
biography       content found
career          content found
achievement     content found
 Stored 15 total chunks for Christo Popov

 Adding Tien Chen Chou profile
biography summary found
biography       con

/home/yezhen/badminton2.0/Badminton_Multi_Agent_Repo/badminton_environment/lib/python3.12/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /home/yezhen/badminton2.0/Badminton_Multi_Agent_Repo/badminton_environment/lib/python3.12/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Disambiguation for 'Brian Yang', trying: Brian Yang (producer)
biography summary found
biography       content found
career          not found, skipping
achievement     not found, skipping
 Stored 1 total chunks for Brian Yang

 Adding Jia Heng Jason Teh profile
biography summary found
biography       content found
career          content found
achievement     content found
 Stored 16 total chunks for Jia Heng Jason Teh

 Adding Chia Hao Lee profile
biography summary found
biography       content found
career          content found
achievement     content found
 Stored 5 total chunks for Chia Hao Lee

 Adding Tzu Wei Wang profile
biography summary found
biography       content found
career          not found, skipping
achievement     content found
 Stored 5 total chunks for Tzu Wei Wang

 Adding Haseena Sunil Prannoy profile
biography summary found
biography       content found
career          content found
achievement     content found
 Stored 18 total chunks for Haseena Sunil Prannoy